# Vie-GameEmo — Training (Simplified)

Notebook này gọi trực tiếp các scripts của project thay vì inline code.

**Pipeline:**
```
import_labels.py → stage0_preprocess.py → transcribe.py → extract_features.py → train.py
```

**Dataset layout cần có trước:**
```
data/
├── raw_videos/train/  ← clips .mp4
├── raw_videos/val/
├── raw_videos/test/
├── labels/train.json  ← [{id, video, choice, confidence}]
├── labels/val.json
└── labels/test.json
```


In [ ]:
# ============================================================
# CELL 1 — Môi trường
# ============================================================
import os, sys
WORKING = os.getcwd()
print(f'Working dir: {WORKING}')

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    print('⚠️  No GPU — training will be very slow')


In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Paths ---
DATASET_INPUT = '/kaggle/input/vie-gameemo-dataset'      # Kaggle input dataset
DATASET_LOCAL = os.path.join(WORKING, 'data')             # local fallback
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'          # project code

# --- Training ---
EPOCHS       = 30
BATCH_SIZE   = 16
FUSION_TYPE  = 'conv_attention_4m'
MIXED_PREC   = 'bf16'

# --- Optional stages ---
TRAIN_COGNITION = False
TRAIN_RLVR      = False


In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
%pip install -q \
    "numpy<2" \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    faster-whisper>=1.0.3 \
    fasttext-wheel \
    scikit-learn \
    pydantic>=2.0 \
    librosa \
    torchvision \
    torchaudio \
    opencv-python-headless \
    tiktoken \
    sentencepiece

In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import shutil, subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
    print(f'Project: Kaggle input → {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/rhy221/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR], check=True)

# Pull latest changes
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    subprocess.run(['git', 'pull'], cwd=PROJECT_DIR, check=True)
    print('Project: pulled latest changes')

# Add to path
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Detect dataset
if os.path.exists(DATASET_INPUT):
    DATA_DIR = DATASET_INPUT
else:
    DATA_DIR = DATASET_LOCAL

SCRIPTS = os.path.join(PROJECT_DIR, 'scripts')
CONFIG  = os.path.join(PROJECT_DIR, 'config.yaml')
print(f'Data:    {DATA_DIR}')
print(f'Scripts: {SCRIPTS}')

## Bước 1 — Import labels + Tiền xử lý


In [ ]:
# ============================================================
# CELL 5 — Import labels → annotations + splits.json
# ============================================================
!python {SCRIPTS}/import_labels.py --data-root {DATA_DIR}


In [ ]:
# ============================================================
# CELL 6 — Tiền xử lý: tách audio + frames từ raw videos
# ============================================================
!python {SCRIPTS}/stage0_preprocess.py \
    --config {CONFIG} \
    --videos-dir {DATA_DIR}/raw_videos \
    --skip-webcam-detect


In [ ]:
# ============================================================
# CELL 7 — ASR: transcribe audio → cập nhật annotations
# ============================================================
!python {SCRIPTS}/transcribe.py --config {CONFIG}


## Bước 2 — Trích xuất features


In [ ]:
# ============================================================
# CELL 8 — Extract + cache features (4 modality encoders)
# ============================================================
!python {SCRIPTS}/extract_features.py --config {CONFIG}


## Bước 3 — Training


In [ ]:
# ============================================================
# CELL 9 — Stage 1: Perception Training
# ============================================================
!python {SCRIPTS}/train.py \
    --config {CONFIG} \
    --stage perception \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --fusion {FUSION_TYPE}


In [ ]:
# ============================================================
# CELL 10 — Eval trên test split
# ============================================================
!python {SCRIPTS}/eval.py --config {CONFIG}


In [ ]:
# ============================================================
# CELL 10b — Phân tích kết quả + gợi ý tinh chỉnh
# ============================================================
# Đọc eval.json → phân tích per-class, confusion, rare class, language gap
# → in ra recommendations cụ thể.
!python {SCRIPTS}/analyze.py \
    --config {CONFIG} \
    --eval-json outputs/results/eval.json \
    --with-fragmentation


In [ ]:
# ============================================================
# CELL 10c — Demo: LLM giải thích cảm xúc (test nhanh)
# ============================================================
# Chạy LLM-1 (post-hoc explainer) trên vài clip test để xem
# chất lượng reasoning trước khi quyết định train Cognition/RLVR.
import torch, json, gc
from pathlib import Path
from types import SimpleNamespace
from vie_gameemo.llm.llm1_explainer import LLM1Explainer
from vie_gameemo.data.schemas import Annotation, EmotionLabel

N_DEMO = 5  # số clip demo
LLM_DEMO_MODEL = 'Qwen/Qwen2.5-7B-Instruct'  # hoặc Qwen3-8B

# Load LLM
prompt_template = (
    'Streamer được dự đoán đang ở trạng thái: {label}.\n'
    'Ngôn ngữ gốc của clip: {source_language}\n\n'
    'Bằng chứng đa phương thức:\n'
    '- Khuôn mặt (Action Units): {face_aus}\n'
    '- Bối cảnh game: {game_context}\n'
    '- Giọng nói: pitch_avg={pitch_hz}Hz, energy={rms_db}dB, shout={shout}\n'
    '- Lời nói: "{transcript}"\n\n'
    'Transcript có thể bằng tiếng Việt hoặc tiếng Anh — hiểu trực tiếp, KHÔNG dịch.\n'
    'Trả lời hoàn toàn bằng tiếng Việt.\n'
    'Hãy giải thích ngắn gọn (dưới 100 từ) vì sao streamer đang ở trạng thái này.'
)

llm = LLM1Explainer(
    model_name=LLM_DEMO_MODEL,
    prompt_template=prompt_template,
    quantization='4bit',
    max_new_tokens=300,
    temperature=0.7,
)
llm.load()

# Load best checkpoint để lấy predictions
from vie_gameemo.fusion import get_fusion
from vie_gameemo.classifiers.mlp import EmotionClassifier
from vie_gameemo.training.perception import load_checkpoint

LABEL_NAMES = [e.value for e in EmotionLabel]
best_ckpt = Path('outputs/checkpoints/perception_best.pt')

if best_ckpt.exists():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fusion = get_fusion('conv_attention_4m', d_model=768, n_modalities=4,
                        return_attention=False).to(device)
    classifier = EmotionClassifier(768, 256, 8, 0.3).to(device)
    load_checkpoint(best_ckpt, fusion, classifier)
    fusion.eval(); classifier.eval()

# Lấy vài clip test có transcript
annot_dir = Path(ANNOT_DIR) if 'ANNOT_DIR' in dir() else Path('data/annotations')
splits_path = Path(SPLITS_PATH) if 'SPLITS_PATH' in dir() else Path('data/splits.json')

with open(splits_path, encoding='utf-8') as f:
    splits = json.load(f)

test_clips = [cid for cid, s in splits.items() if s == 'test'][:N_DEMO * 3]
demo_clips = []
for cid in test_clips:
    ann_path = annot_dir / f'{cid}.json'
    if ann_path.exists():
        ann = json.loads(ann_path.read_text(encoding='utf-8'))
        if ann.get('transcript', '').strip():
            demo_clips.append(ann)
    if len(demo_clips) >= N_DEMO:
        break

if not demo_clips:
    # Fallback: lấy bất kỳ clip nào có transcript
    for p in sorted(annot_dir.glob('*.json'))[:50]:
        ann = json.loads(p.read_text(encoding='utf-8'))
        if ann.get('transcript', '').strip():
            demo_clips.append(ann)
        if len(demo_clips) >= N_DEMO:
            break

print(f'Demo LLM explanation trên {len(demo_clips)} clips:\n')
print('=' * 70)

for ann in demo_clips:
    evidence = {
        'label': ann.get('emotion_label', 'neutral'),
        'face_aus': ann.get('face_aus', 'N/A'),
        'game_context': ann.get('visual_objective_desc', 'N/A'),
        'pitch_hz': 0,
        'rms_db': 0,
        'shout': False,
        'transcript': ann.get('transcript', ''),
        'source_language': ann.get('source_language', 'vi'),
    }
    result = llm.reason(evidence)

    print(f'Clip: {ann["clip_id"]}')
    print(f'Label: {ann.get("emotion_label", "?")} | Lang: {ann.get("source_language", "vi")}')
    print(f'Transcript: "{ann.get("transcript", "")[:100]}"')
    print(f'\nLLM Reasoning:')
    print(f'  {result.reasoning}')
    print(f'  → Answer: {result.answer} (format_valid={result.format_valid})')
    print('-' * 70)

llm.unload()
del llm; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n✅ Demo xong. Xem reasoning có mạch lạc không trước khi train Cognition.')


## Bước 4 — (Tùy chọn) Cognition + RLVR


In [ ]:
# ============================================================
# CELL 11 — Stage 2: Cognition (joint LLM + adapter)
# ============================================================
if TRAIN_COGNITION:
    !python {SCRIPTS}/train.py \
        --config {CONFIG} \
        --stage cognition \
        --resume-from outputs/checkpoints/perception_best.pt
else:
    print('TRAIN_COGNITION=False — bỏ qua')


In [ ]:
# ============================================================
# CELL 12 — RLVR (LLM-4, tùy chọn)
# ============================================================
if TRAIN_RLVR:
    !python {SCRIPTS}/train_rlvr.py \
        --config {CONFIG} \
        --phase cold-start \
        --base-model Qwen/Qwen2.5-1.5B-Instruct \
        --epochs 2
else:
    print('TRAIN_RLVR=False — bỏ qua')


In [ ]:
# ============================================================
# CELL 13 — Lưu checkpoint để download
# ============================================================
import zipfile
from pathlib import Path

CKPT_DIR = 'outputs/checkpoints'
ckpt_files = list(Path(CKPT_DIR).glob('*.pt'))
print(f'Checkpoints ({len(ckpt_files)}):')
for f in ckpt_files:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

archive = os.path.join(WORKING, 'vie_gameemo_checkpoints.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ckpt in ckpt_files:
        zf.write(str(ckpt), f'checkpoints/{ckpt.name}')
    zf.write(CONFIG, 'config.yaml')

print(f'\n✅ Archive: {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')
